# 3 · opencode — a coding agent on NRP

*Adapted from Daniel Diaz's CLARIPHY lesson [Agentic Workflows](https://training.nrp-nautilus.io/clariphy/3_agentic.html) ([nrp-nautilus/nrp-training](https://github.com/nrp-nautilus/nrp-training/tree/materials/clariphy)).*

**The goal for today:** `opencode` starts in your JupyterLab terminal and answers you with both models. That is all the setup the hackathon needs. Then you have it write a script, and you check its work.

Run cells with **Shift+Enter**. Cells share one persistent shell, so `export` and `cd` carry from cell to cell.

In [notebook 1](1_talk_to_a_model.ipynb) you **called** NRP's managed LLMs from Python, and in [notebook 2](2_agent_by_hand.ipynb) you ran the agent loop by hand. Now you will point an **agentic coding tool** — that same loop, with file and shell tools — at the same endpoint and have it plan, write, and run code autonomously on your behalf.

The key teaching point is **portability**: anything that speaks an OpenAI-compatible `base_url` works against NRP, so the agentic workflow you already use locally runs unchanged against NRP's managed inference — no API key handoff theater, no per-user billing.

> **Where to run what:** the setup steps below are notebook cells (Shift+Enter). `opencode` itself is an interactive
> program, so it runs in a JupyterLab terminal: those steps are marked 🖥️ **Terminal step**.
>
> Nothing here edits your shell startup files or any opencode configuration you already have: the exercise lives in
> a new `~/opencode-exercise` folder, and your key stays in the private file notebook 1 saved.

## Set up opencode

[`opencode`](https://opencode.ai) is an open-source terminal UI agentic coding assistant — similar in spirit to Claude Code or Cursor's CLI. It reads your project files, plans changes, edits code, and iterates.

### Install

If you already have opencode, this keeps your copy; otherwise it runs the official installer, which puts `opencode` in `~/.opencode/bin`:

In [ ]:
export PATH="$HOME/.opencode/bin:$PATH"
command -v opencode >/dev/null || curl -fsSL https://opencode.ai/install | bash -s -- --no-modify-path
opencode --version


### Create the exercise folder

opencode scopes file writes to the nearest `.git` directory, not simply your
shell's current directory — without one, it can fall back to a much wider
default and write generated files somewhere you don't expect (a
[known opencode behavior](https://github.com/anomalyco/opencode/issues/15192),
not something specific to this training). So the exercise gets its own fresh
folder, turned into a git project:

In [ ]:
mkdir -p ~/opencode-exercise && cd ~/opencode-exercise
git init -q
pwd


### Configure NRP as the provider

opencode needs two things: where NRP's endpoint is, and your key. Notebook 1 already saved the key to
`~/.nrp-llm-token`, a private file in your home directory. Terminals in JupyterLab are **separate processes**
from this notebook, so `export`s here would not reach them; the file is how opencode will find the key (the
config below points at it). It sits outside the project, so it never ends up in the agent's working files or in
git. Check that it is there:

In [ ]:
if [ -s ~/.nrp-llm-token ]; then
    echo "Key found in ~/.nrp-llm-token"
else
    echo "No key yet: run the key cell at the top of notebook 1 (1_talk_to_a_model.ipynb), then this cell again."
fi

Then write the NRP provider as a **project config** — an `opencode.json` inside the exercise folder. opencode merges it with any global config you have, with the project taking priority, so your own `~/.config/opencode` settings are left alone and this only applies inside `~/opencode-exercise`. (See the [full client-config reference](https://nrp.ai/documentation/userdocs/ai/llm-managed/client-configs/) for opencode, VS Code, Claude Code, and more.)

In [ ]:
cd ~/opencode-exercise
cat > opencode.json <<'JSON'
{
  "$schema": "https://opencode.ai/config.json",
  "provider": {
    "nrp": {
      "npm": "@ai-sdk/openai-compatible",
      "name": "NRP LLM",
      "options": {
        "baseURL": "https://ellm.nrp-nautilus.io/v1",
        "apiKey": "{file:~/.nrp-llm-token}"
      },
      "models": {
        "deepseek-v4-flash": { "name": "DeepSeek V4 Flash" },
        "glm-5":             { "name": "GLM-5"             },
        "gpt-oss":           { "name": "GPT-OSS"           },
        "qwen3":             { "name": "Qwen3 397B"        }
      }
    }
  },
  "model": "nrp/deepseek-v4-flash"
}
JSON
cat opencode.json

# opencode 2 keeps a background service that remembers old settings and keys: stop it, so the next
# `opencode` start reads this file and your current key (opencode 1.x has no such service: nothing happens)
if opencode service stop > /dev/null 2>&1; then
    echo "opencode: background service stopped (the next start reads these settings)"
fi


`{file:~/.nrp-llm-token}` tells opencode to read your key from that file when it starts — so opencode works in **any** terminal, with nothing to export first — and with *your* key, even on the
training hub, where a shared `OPENAI_API_KEY` may already be set.

> **Switching models:** inside opencode, type `/models`, then pick one, to change the active model mid-session. Try the same task with `deepseek-v4-flash` and `glm-5`: same tools, same files, different model — compare what each one does.

### Python environment and the data file — before you start opencode

Two things the exercise needs no matter what the agent writes: a private Python environment for the project
(a **virtual environment**, `.venv`, so installs cannot clash with the system Python) with `uproot` and
`tabulate` in it, and a real CMS NanoAOD file to test on — simulated collisions in the CMS NanoAOD format,
~20 MB from NRP's storage. Run both cells now:

In [ ]:
cd ~/opencode-exercise
python3 -m venv .venv
for f in '.venv/' '*.root'; do grep -qsxF "$f" .gitignore || echo "$f" >> .gitignore; done   # out of git and the agent's searches
.venv/bin/python -m pip install -q 'uproot>=5' tabulate && echo "environment ready"

In [ ]:
cd ~/opencode-exercise

# A CMS NanoAOD file from NRP's S3 storage — no proxy, no credentials (~20 MB).
[ -f nanoout_1.root ] || curl -fsSL -o nanoout_1.root \
  "https://s3-west.nrp-nautilus.io/transfer-bucket/QCD_Bin-Pt-15to7000_TuneCP5_13p6TeV_pythia8_nano.root" \
  || rm -f nanoout_1.root

ls -lh nanoout_1.root


### What opencode sends the model

opencode is notebook 2's loop with more tools. Every turn it sends the model a `messages` list like the
ones you built by hand: its own long **system prompt** (how to behave, how to edit files), the **schemas
of its tools** (read, edit, bash, search, …), your request, and every tool call and result so far. Your
project files reach the model only through those tool results: it reads what it chooses to read.

That is why the context fills up in a long session (`/compact` summarises it), why the folder matters,
and why the same prompt behaves differently with `deepseek-v4-flash` and `glm-5`. Watch for those tool calls while it works.

### Exercise: Build a CMS analysis helper

🖥️ **Terminal step** — `opencode` is an interactive terminal UI, so launch it from a JupyterLab terminal (**File → New → Terminal**), not this notebook:

```bash
cd ~/opencode-exercise
export PATH="$HOME/.opencode/bin:$PATH"
opencode
```

Getting `Forbidden` or `401` responses once inside opencode? It could not use your key. Quit opencode (`/exit`), then in the terminal run `rm ~/.nrp-llm-token` (no `!` here: that is only for notebook cells), rerun notebook 1's key cell, run `opencode service stop`, and start `opencode` again: its background service remembers the old key until it stops.

The prompt is active as soon as opencode opens. First type `hi` and press Enter: you should get a reply. Then type `/models`, pick `glm-5`, and say `hi` again. A reply from both models is the setup the hackathon needs. (`/` opens the command menu for things like `/models` or `/clear`, not the prompt itself.)

Now paste the following task:

```text
Write a Python script cms_nano_summary.py that uses the uproot library to open
a CMS NanoAOD ROOT file and print a summary of its contents.

The input is a real CMS NanoAOD file whose path is given as a command-line
argument, positional rather than an option:
  python cms_nano_summary.py nanoout_1.root
It has an "Events" TTree using the standard NanoAOD flat-branch
convention: collections appear as "<Collection>_<variable>" (Muon_pt,
Muon_eta, Jet_pt, ...), with an "n<Collection>" counter branch giving the
per-event multiplicity of each collection.

The script should:
- Group the branches by collection (all "Muon_*" together, all "Jet_*"
  together, and so on), listing anything that isn't part of a collection
  under "Event-level".
- For each branch print its name, type, and title/description if uproot
  exposes one.
- Print the total number of events at the end.
- Have a proper argparse interface and a top-level docstring.

Also write a requirements.txt pinning uproot>=5 and tabulate.
```

> **Important:** if opencode generates a file named `uproot.py`, rename it — it would shadow the `uproot` package on import.

When the agent says it is done, install anything extra it asked for and check that its script at least starts.
Calling `.venv/bin/python` uses the project's environment directly, so there is nothing to activate. (Agent not
done after ~20 min? The fallback cell is a little further down.)

In [ ]:
cd ~/opencode-exercise
[ -f requirements.txt ] && .venv/bin/python -m pip install -q -r requirements.txt
.venv/bin/python cms_nano_summary.py --help

In [ ]:
cd ~/opencode-exercise
# The first 40 lines. A "Broken pipe" error after them is head stopping early, not a problem with the file.
.venv/bin/python cms_nano_summary.py nanoout_1.root | head -40


This is the real test of the agent's work: does the script actually survive contact with a NanoAOD file? Common ways a first attempt falls over — worth feeding straight back to opencode rather than fixing by hand:

- Treating every branch as flat when the jagged collection branches need `n<Collection>` to interpret.
- Crashing on branches with no title instead of printing a blank description.
- Assuming a fixed set of collections rather than discovering them from the file.

If it fails, paste the traceback into opencode and let it debug — watching an agent iterate on a real error is the point of the exercise.

### Agent not done after ~20 min, or its script still fails? Run this instead of waiting

The point of this exercise is the checking. This cell puts the reference script in place of a script that is
missing or fails on the file, and keeps the agent's as `cms_nano_summary.agent.py`. If the agent's script runs, the
cell leaves it alone.

In [ ]:
cd ~/opencode-exercise
if [ -f cms_nano_summary.py ] && timeout 120 .venv/bin/python cms_nano_summary.py nanoout_1.root > /dev/null 2>&1; then
    echo "Your agent's script runs on the file: keep it, and go on to the check."
else
    [ -f cms_nano_summary.py ] && cp cms_nano_summary.py cms_nano_summary.agent.py && echo "the agent's script is kept as cms_nano_summary.agent.py"
    { cp ~/hsf-india-agents/fallback/cms_nano_summary.py . 2>/dev/null \
      || curl -fsSL https://raw.githubusercontent.com/andrzejnovak/hsf-india-agents/main/fallback/cms_nano_summary.py -o cms_nano_summary.py; } \
      && echo "reference script copied: go on to the check" || echo "could not copy the reference script"
fi

### Check the agent's work — a partial check

The script ran — but is its output **right**? Here the file itself is the planted truth: we know exactly which
columns it contains. This cell saves the script's output to `summary.txt` and compares it with the file: is every
column named, and do four collections (Muon, Jet, Electron, boostedTau) get a heading of their own? It is a
**partial** check — it does not test that each column appears once or sits in the right group — so look at the
grouping yourself too.

In [ ]:
cd ~/opencode-exercise
# 1) the agent's script on the real file; its output goes to summary.txt
.venv/bin/python cms_nano_summary.py nanoout_1.root > summary.txt 2>&1; echo "the script exited with status $?"

# 2) the check: compare a summary with the file itself (defined once, used again below)
check_summary() {
.venv/bin/python - "$1" <<'PY'
import re, sys, uproot
keys = uproot.open("nanoout_1.root")["Events"].keys()
out = open(sys.argv[1]).read()
words = set(re.findall(r"\w+", out))        # whole names: nMuon is not found inside nMuon_whatever
missing = [k for k in keys if k not in words]
print(f"{sys.argv[1]}: {len(keys)} columns in the file, {len(missing)} missing from the summary", missing[:5])
for group in ["Muon", "Jet", "Electron", "boostedTau"]:
    # a heading line: "[Muon]", "Muon (78 branches)", "Muon_*", "Collection: Muon"; not a column like "Muon_pt"
    has_header = bool(re.search(rf"^\W*(?:[Cc]ollection\W*)?{group}(?:_\*)?(?!\w)", out, re.M))
    print(f"  {group:11s} has its own group:", "yes" if has_header else "NO - look at the summary")
PY
}
check_summary summary.txt

**Does the check itself work?** A check that never fails proves nothing. Break the summary on purpose and run it
again: it must complain. (This uses `check_summary` from the cell above: after a kernel restart, run that cell first.)

In [ ]:
cd ~/opencode-exercise
# Prove the check works: remove every line that mentions Muon, check again. It must complain.
grep -v Muon summary.txt > broken.txt
if type check_summary > /dev/null 2>&1; then check_summary broken.txt
else echo "Run the check cell above first: it defines check_summary (a kernel restart forgets it)."; fi

**Things to try (back in the opencode terminal):**
- Once the script is written, ask opencode to add a `--filter` argument that limits output to a specific collection (e.g., `--filter Muon`).
- Switch to `glm-5` and ask it to add unit tests with `pytest`.

## Before Wednesday

`opencode` should start in your JupyterLab terminal and answer with both models: that is all the setup the hackathon
needs. The NRP settings are in `~/opencode-exercise/opencode.json` and apply inside that folder. For a project in
another folder, copy them there and make it a git project:

```bash
mkdir -p ~/my-project && cp ~/opencode-exercise/opencode.json ~/my-project/ && cd ~/my-project && git init -q
```